In [2]:
from typing import TypedDict,Annotated
from langgraph.types import Send
import operator
from langgraph.graph import START,END,StateGraph,MessagesState
from langgraph.graph.message import add_messages
from langchain_core.messages import AIMessage,HumanMessage,ToolMessage
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env")

True

In [5]:
class GlobalState(TypedDict):
    items: list
    results: Annotated[list,operator.add]

class ItemState(TypedDict):
    item: str

def process_state(state:ItemState)->dict:
    item=state['item']
    print(f"Processing Data : {state['item']}")
    result=f"Processed:{state['item']}"
    return {"results": [result]}

def distribute_load(state:GlobalState)->dict:
    print(f"Distributing {state['items']} loads for processing")
    return [Send("process",{"item":item}) for item in state['items']]

graph=StateGraph(GlobalState)
graph.add_node("process",process_state)
graph.add_conditional_edges(START,distribute_load,["process"])
graph.add_edge("process",END)

app=graph.compile()
result=app.invoke({
    "items":["apple", "banana", "cherry", "date"],
    "results":[]
})

print(result)


Distributing ['apple', 'banana', 'cherry', 'date'] loads for processing
Processing Data : apple
Processing Data : banana
Processing Data : cherry
Processing Data : date
{'items': ['apple', 'banana', 'cherry', 'date'], 'results': ['Processed:apple', 'Processed:banana', 'Processed:cherry', 'Processed:date']}


In [18]:
class GlobalState(TypedDict):
    documents: list[str]
    reviews: Annotated[list[str],operator.add]
    summary: str

class NodeState(TypedDict):
    documents: str
    doc_index: int

def load_documents(state:GlobalState)->dict:
    """Loads 4 documents into Global State:"""
    docs=[
         "Q4 Revenue Report: revenue up 18% YoY",
        "Risk Assessment: supply chain risks identified",
        "Product Roadmap: 3 major features planned for H1",
        "HR Update: headcount grew by 15 in Q4",
    ]
    print(f"Documents of {len(docs)} are loaded")
    return {"documents":docs}

def review_documents(state:NodeState)->dict:
    """
    Worker node — receives ONE document via Send().
    Note: 'state' here is WorkerState, NOT OverallState.
    Each parallel invocation of this node has a different 'document'.
    """

    doc=state['documents']
    index=state['doc_index']
    review=f"Reviewing DOC {index+1}: {doc[:40]}"
    print(f"DOCS {index+1} Reviewed")
    return {"reviews":[review]}

def summary_creation(state:GlobalState)->dict:
    """Fan-in node: receives all reviews merged by the reducer."""
    print(f"Documents collected : {len(state['reviews'])} , Generating summary ...")
    summary="Summary :".join(state['reviews'])

    return {"summary":summary}

def dispatch_reviewers(state:GlobalState)->dict:
    """Called as a conditional edge function.
    Returns N Send objects → spawns N parallel review_document workers,
    each with its OWN custom state dict (different document + index).
    """
    return [Send("review",{"documents":doc,"doc_index":i}) for i,doc in enumerate(state['documents'])]

graph=StateGraph(GlobalState)
graph.add_node("load",load_documents)
graph.add_node("review",review_documents)
graph.add_node("summary",summary_creation)
graph.add_node("dispatch",dispatch_reviewers)

graph.add_edge(START,"load")
graph.add_conditional_edges("load",dispatch_reviewers,["review"])
graph.add_edge("review","summary")
graph.add_edge("summary",END)

app=graph.compile()

result=app.invoke({"documents":[],"reviews":[],"summary":""})
print(result)

Documents of 4 are loaded
DOCS 1 Reviewed
DOCS 2 Reviewed
DOCS 3 Reviewed
DOCS 4 Reviewed
Documents collected : 4 , Generating summary ...
{'documents': ['Q4 Revenue Report: revenue up 18% YoY', 'Risk Assessment: supply chain risks identified', 'Product Roadmap: 3 major features planned for H1', 'HR Update: headcount grew by 15 in Q4'], 'reviews': ['Reviewing DOC 1: Q4 Revenue Report: revenue up 18% YoY', 'Reviewing DOC 2: Risk Assessment: supply chain risks iden', 'Reviewing DOC 3: Product Roadmap: 3 major features planne', 'Reviewing DOC 4: HR Update: headcount grew by 15 in Q4'], 'summary': 'Reviewing DOC 1: Q4 Revenue Report: revenue up 18% YoYSummary :Reviewing DOC 2: Risk Assessment: supply chain risks idenSummary :Reviewing DOC 3: Product Roadmap: 3 major features planneSummary :Reviewing DOC 4: HR Update: headcount grew by 15 in Q4'}


In [ ]:
class AgentState(TypedDict):
    task: str
    result: str

def research_node(state:AgentState)->dict:
    result=f"Research doing  on Topic: {state['task']}"
    print(f"Research Done")
    return {"resutlt":result}

def analyze_node(state:AgentState)->dict:
    result=f"Analyzing doing on Topic: {state['task']}"
    print(f"Analyzing Done")
    return {"result": result}

research_graph=StateGraph(AgentState)
research_graph.add_node("research",research_node)
research_graph.add_edge(START,"research")
research_graph.add_edge("research",END)

analyze_graph=StateGraph(AgentState)
analyze_graph.add_node("analyze",analyze_node)
analyze_graph.add_edge(START,"analyze")
analyze_graph.add_edge("analyze",END)

research_app=research_graph.compile()
analyze_app=analyze_graph.compile()

['Q4 Revenue Report: revenue up 18% YoY', 'Risk Assessment: supply chain risks identified', 'Product Roadmap: 3 major features planned for H1', 'HR Update: headcount grew by 15 in Q4']


In [30]:
class PreProcessingState(TypedDict):
    text: str
    characters: int

def clean_text(state:PreProcessingState)->dict:
    cleaned_text=state['text'].strip().lower()
    print(f"CLEANED TEXT: {cleaned_text}")
    return {"text":cleaned_text}

def count_characters(state:PreProcessingState)->dict:
    characters=len(state['text'])
    print(f"TEXT CHARACTERS: {characters}")
    return {"characters": characters}

graph=StateGraph(PreProcessingState)
graph.add_node("clean",clean_text)
graph.add_node("count",count_characters)
graph.add_edge(START,"clean")
graph.add_edge("clean","count")
graph.add_edge("count",END)

pre_app=graph.compile()

class AnalysisState(TypedDict):
    text: str
    keywords: list[str]

def extract_keywords(state:AnalysisState)->dict:
    keyword_search=[w for w in state["text"].split() if len(w)>5]
    print(f"keywords:{keyword_search[:5]}")
    return {"keywords": keyword_search}

def analyze_sentiment(state:AnalysisState)->dict:
    good_words=["great", "good", "excellent", "amazing", "fantastic"]
    sentiment_score=sum(1 for w in state["keywords"] if w in good_words)
    if sentiment_score>0:
        label="positive"
    else:
        label="neutral"

    return {"text": f"SENTIMENT:{label}: TEXT:{state['text']}"}

graph2=StateGraph(AnalysisState)
graph2.add_node("extract",extract_keywords)
graph2.add_node("analyze",analyze_sentiment)
graph2.add_edge(START,"extract")
graph2.add_edge("extract","analyze")
graph2.add_edge("analyze",END)

ana_app=graph2.compile()

class ParentState(TypedDict):
    text: str
    result: str

def format_result(state:ParentState)->dict:
    print(f"FORMATTING RESULT...")
    return {"result":state['text']}

parent_graph=StateGraph(ParentState)
parent_graph.add_node("format",format_result)
parent_graph.add_node("prepreocessing",pre_app)
parent_graph.add_node("analysis",ana_app)

parent_graph.add_edge(START,"prepreocessing")
parent_graph.add_edge("prepreocessing","analysis")
parent_graph.add_edge("analysis","format")
parent_graph.add_edge("format",END)

app=parent_graph.compile()

result=app.invoke({"text":"  LangGraph is Great for building Agentic AI systems!  ","result":""})
print(result['result'])

CLEANED TEXT: langgraph is great for building agentic ai systems!
TEXT CHARACTERS: 51
keywords:['langgraph', 'building', 'agentic', 'systems!']
FORMATTING RESULT...
SENTIMENT:neutral: TEXT:langgraph is great for building agentic ai systems!


In [10]:
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI

class AgentState(TypedDict):
    messages: Annotated[list,add_messages]
    research_results: list
    analysis_results: list
    final_report: str

research_llm=ChatOpenAI(model='gpt-4o-mini',temperature=0.3)
analyser_llm=ChatOpenAI(model='gpt-4o-mini',temperature=0)
synthesizer_llm=ChatOpenAI(model='gpt-4o-mini',temperature=0.5)

def coordinator(state:AgentState)->list[Send]:
    query=state['messages'][-1].content
    print(f"Co-ordinating Task: Distributing Query {query}")

    return [ Send("researcher",state),Send("analyzer",state)]

def researcher(state:AgentState)->dict:
    query=state['messages'][-1].content
    print(f"Doing Research")
    response=research_llm.invoke([HumanMessage(content=f"Research this topic and provide key facts:{query}")])
    return {'research_results':[response.content]}

def analyzer(state:AgentState)->dict:
    query=state['messages'][-1].content
    print(f"Doing Analyzation")
    response=analyser_llm.invoke([HumanMessage(content=f"Analyze the credibility and relevance of information about:{query}")])
    return {'analysis_results':[response.content]}

def synthesizer(state:AgentState)->dict:
    print(f"Synthesizing Final content")
    research="\n".join(state.get("research_results",[]))
    analyze="\n".join(state.get("analysis_results",[]))

    final_result=synthesizer_llm.invoke([HumanMessage(content=f"Synthesize these findings into a coherent report:\n\nResearch:\n{research}\n\nAnalysis:\n{analyze}")])
    print(f"FINAL RESULT:{final_result}")
    return {"final_report":final_result.content}


graph=StateGraph(AgentState)
graph.add_node("researcher",researcher)
graph.add_node("analyzer",analyzer)
graph.add_node("synthesizer",synthesizer)

graph.add_conditional_edges(START,coordinator,["researcher","analyzer"])
graph.add_edge("researcher","synthesizer")
graph.add_edge("analyzer","synthesizer")
graph.add_edge("synthesizer",END)

app=graph.compile()

print("Multiagent Built")


Multiagent Built


In [11]:
result=app.invoke({"messages":[HumanMessage(content="Explain the impact of quantum computing on cryptography")],"research_results":[],"analysis_results":[],"final_report":""})
print(result)

Co-ordinating Task: Distributing Query Explain the impact of quantum computing on cryptography
Doing Research
Doing Analyzation
Synthesizing Final content
FINAL RESULT:content="# Impact of Quantum Computing on Cryptography: A Comprehensive Report\n\n## Introduction\nQuantum computing is poised to revolutionize various fields, with cryptography being one of the most significantly affected areas. Its ability to solve specific mathematical problems at unprecedented speeds threatens the security of current cryptographic systems while also paving the way for innovative solutions. This report synthesizes key findings on the implications of quantum computing for cryptography, addressing both the challenges it presents and the opportunities it creates.\n\n## Quantum Algorithms and Their Implications\n### 1. Shor's Algorithm\nDeveloped by Peter Shor in 1994, Shor's Algorithm can factor large integers exponentially faster than classical algorithms. This capability directly threatens widely used 

In [14]:
from langgraph.types import Command
from typing import Literal
import operator
class AgentState(TypedDict):
    task: str
    log: Annotated[list[str],operator.add]
    result: str

class CodeAgent(TypedDict):
    task: str
    log: Annotated[list[str],operator.add]
    result: str

class WebState(TypedDict):
    task: str
    log: Annotated[list[str],operator.add]
    result: str

def web_search(state:WebState):
    print(f"WEB AGENT Searching")
    return {"log":["Web search completed"]}

def web_summarize(state:WebState):
    print(f"Summarizing the web results for :{state['task']}")
    return {"result":"Web results","log":["Web Results summarized"]}

web_graph=StateGraph(WebState)
web_graph.add_node("search",web_search)
web_graph.add_node("summarize",web_summarize)
web_graph.add_edge(START,"search")
web_graph.add_edge("search","summarize")
web_graph.add_edge("summarize",END)

web_agent=web_graph.compile()

def write_code(state:CodeAgent):
    print(f"Code writing is complete")
    return {"log":["Wrting code is complete"]}

def test_code(state:CodeAgent):
    print(f"Code Testing is complete")
    return {"result":f"Final code for task:{state['task']}","log":["Testing code is complete"]}

code_graph=StateGraph(CodeAgent)
code_graph.add_node("write",write_code)
code_graph.add_node("test",test_code)
code_graph.add_edge(START,"write")
code_graph.add_edge("write","test")
code_graph.add_edge("test",END)

code_agent=code_graph.compile()


def orchestrator(state:AgentState)->Command[Literal["web_agent","code_agent"]]:
    print(f"Orchestrating Agent for Task:{state['task']}")
    task=state['task'].lower()
    if "code" in task or "code" in task or "python" in task:
        print(f"-> Routing to Code Agent")
        return Command(update={"log":["Routing to Code Agent"]},goto="code_agent")
    else:
        print(f"-> Routing to Web Agent")
        return Command(update={"log":["Routing to Web Agent"]},goto="web_agent")

def synthesizer(state:AgentState):
    print(f"Result for TASK {state['task']}:{state['result']}")
    print(f"Full Logs:{state['log']}")
    return {}

agent_graph=StateGraph(AgentState)
agent_graph.add_node("orchestrator",orchestrator)
agent_graph.add_node("synthesizer",synthesizer)
agent_graph.add_node("code_agent",code_agent)
agent_graph.add_node("web_agent",web_agent)

agent_graph.add_edge(START,"orchestrator")
agent_graph.add_edge("code_agent","synthesizer")
agent_graph.add_edge("web_agent","synthesizer")
agent_graph.add_edge("synthesizer",END)

app=agent_graph.compile()

In [15]:
result=app.invoke({"task": "Write a Python function to sort a list", "log": [], "result": ""})

Orchestrating Agent for Task:Write a Python function to sort a list
-> Routing to Code Agent
Code writing is complete
Code Testing is complete
Result for TASK Write a Python function to sort a list:Final code for task:Write a Python function to sort a list
Full Logs:['Routing to Code Agent', 'Routing to Code Agent', 'Wrting code is complete', 'Testing code is complete']
